# Day 3 — The agent loop, budgets and planning (Gemini)
**Theory:** [day-03-agent-loop-and-planning.md](../theory/day-03-agent-loop-and-planning.md) · ⏱ ~75 min · Level: medium

**You will:**
1. Write a production-style agent loop from scratch (stop conditions, budgets, errors)
2. Add a `todo_write` planning tool and watch the plan evolve
3. Compare `thinking_level` low vs high on cost, steps and quality
4. Add repeat detection

In [ ]:
# ⚙️ Setup: works in GitHub Codespaces, locally, in Colab and on Kaggle. Run this cell first.
# Please don't let an AI "auto-fix" rewrite the imports below: the course packages are `course_utils` and `shopease`.
import os, sys, pathlib, subprocess
IN_COLAB, IN_KAGGLE = "google.colab" in sys.modules, os.path.exists("/kaggle/working")
if IN_COLAB or IN_KAGGLE:  # clone (or update) the course repo, install it, read the key from notebook secrets
    BRANCH = os.getenv("COURSE_BRANCH", "main")
    REPO_URL = os.getenv("COURSE_REPO_URL", "https://github.com/Rahulverma5/agentic-ai-course")
    REPO = pathlib.Path(os.getenv("COURSE_BASE_DIR", "/content" if IN_COLAB else "/kaggle/working")) / "agentic-ai-course"
    if not (REPO / ".git").exists():
        subprocess.run(["git", "clone", "-q", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
    else:  # an older clone from a previous session: bring it up to date
        subprocess.run(["git", "-C", str(REPO), "fetch", "-q", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "checkout", "-q", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "reset", "-q", "--hard", f"origin/{BRANCH}"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt"), "-e", str(REPO)],
                   check=True)
    try:
        if IN_COLAB:
            from google.colab import userdata
            os.environ.setdefault("GEMINI_API_KEY", userdata.get("GEMINI_API_KEY"))
        else:
            from kaggle_secrets import UserSecretsClient
            os.environ.setdefault("GEMINI_API_KEY", UserSecretsClient().get_secret("GEMINI_API_KEY"))
    except Exception as e:
        print("Add GEMINI_API_KEY to the notebook secrets (🔑 icon in Colab, Add-ons → Secrets in Kaggle):", e)
else:  # Codespaces / local: use the checked-out repo
    REPO = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path[:0] = [str(REPO), str(REPO / "projects" / "shopease-support-agent")]
os.chdir(REPO)  # every path in this notebook is relative to the repo root
try:
    from google.genai import types  # Gemini SDK types: Content, Part, GenerateContentConfig, ...
    from course_utils import check_setup, client, MODEL, CHEAP_MODEL, ask, tracker, text_of, thinking, user
    from shopease.data import setup_data
except ImportError as e:
    raise ImportError(
        f"{e}\n\nThe course code at {REPO} is missing or outdated. Fix it like this:\n"
        f"  • Colab/Kaggle: run  !rm -rf {REPO}  then Runtime → Restart session, and run this cell again.\n"
        "  • Codespaces/local: run  git pull && pip install -e .  in a terminal, then restart the kernel.") from None
setup_data(quiet=True)  # creates the ShopEase sample data if it's missing
check_setup();

## 1. Tools: reuse the ShopEase tools
They're scoped to one logged-in customer through a `Session`, as in production.

In [ ]:
import json, time
from shopease.tools import Session, TOOLS, dispatch
from course_utils import to_gemini_tools, function_response

session = Session(customer_id="C001")
print([t["name"] for t in TOOLS])
print(dispatch("get_customer_overview", {}, session)[0][:300])

## 2. The loop, from scratch
Think → act → observe, plus three budgets (steps, $, seconds). Read every line. This is the core of every agent.

In [ ]:
def my_agent(user_msg, tools, run, system="", max_steps=12, max_cost=0.30, max_seconds=180, level="medium"):
    contents, t0, cost0 = [user(user_msg)], time.time(), tracker.total()
    config = types.GenerateContentConfig(system_instruction=system or None, tools=to_gemini_tools(tools),
                                         thinking_config=thinking(level))
    for step in range(1, max_steps + 1):
        r = client.models.generate_content(model=MODEL, contents=contents, config=config)   # THINK
        tracker.add(MODEL, r.usage_metadata, f"step {step}")
        contents.append(r.candidates[0].content)
        if not r.function_calls:                                                             # DONE?
            finish = r.candidates[0].finish_reason
            return (text_of(r) if finish.name == "STOP" else f"[stopped: {finish.name}]"), step, contents
        parts = []
        for fc in r.function_calls:                                                          # ACT
            out, err = run(fc.name, dict(fc.args or {}))
            print(f"  step {step} 🔧 {fc.name}({json.dumps(dict(fc.args or {}))[:70]}){' ❌' if err else ''}")
            parts.append(function_response(fc, out, err))
        contents.append(types.Content(role="user", parts=parts))                            # OBSERVE
        if tracker.total() - cost0 > max_cost:
            return "[stopped: cost budget]", step, contents
        if time.time() - t0 > max_seconds:
            return "[stopped: time budget]", step, contents
    return "[stopped: step limit]", max_steps, contents

SYSTEM = "You are ShopEase support. Use tools before answering. Always check refund eligibility before promising refunds."
run = lambda name, args: dispatch(name, args, session)
answer, steps, transcript = my_agent("Can I get a refund for my coffee grinder?", TOOLS, run, SYSTEM)
print(f"\n({steps} steps)\n{answer}")

## 3. Planning with a todo list
A `todo_write` tool lets the model keep its plan near the *end* of the context, where attention is strongest.

In [ ]:
TODOS = []
def todo_write(items):
    TODOS[:] = items
    return "\n".join(f"[{ {'pending': ' ', 'in_progress': '~', 'done': 'x'}[i['status']] }] {i['task']}" for i in items)

TODO_TOOL = {"name": "todo_write", "description": "Write/replace your task list. Use it for multi-step tasks; "
             "update statuses as you go.",
             "input_schema": {"type": "object", "properties": {"items": {"type": "array", "items": {
                 "type": "object", "properties": {"task": {"type": "string"},
                                                  "status": {"type": "string", "enum": ["pending", "in_progress", "done"]}},
                 "required": ["task", "status"]}}}, "required": ["items"]}}

def run_with_todo(name, args):
    if name == "todo_write":
        out = todo_write(**args); print("   📝\n" + "\n".join("      " + l for l in out.splitlines()))
        return out, False
    return dispatch(name, args, session)

task = ("For each of my orders that is delivered, tell me if it can still be returned, "
        "then summarise everything in a short table.")
answer, steps, _ = my_agent(task, TOOLS + [TODO_TOOL], run_with_todo, SYSTEM)
print(f"\n({steps} steps)\n{answer}")

## 4. Thinking level: low vs high
Same task, two thinking levels. More thinking = more (billed) thinking tokens and latency, sometimes better answers.

In [ ]:
import pandas as pd
rows = []
for level in ("low", "high"):
    before, t0, n0 = tracker.total(), time.time(), len(tracker.calls)
    ans, steps, _ = my_agent("Which of my orders can I still return, and why?", TOOLS, run, SYSTEM, level=level)
    rows.append({"thinking": level, "steps": steps, "seconds": round(time.time() - t0, 1),
                 "thinking tokens": sum(c["thinking"] for c in tracker.calls[n0:]),
                 "≈cost $": round(tracker.total() - before, 4), "answer": ans[:120]})
pd.DataFrame(rows)

## 5. Repeat detection
A common failure: the same tool call with the same input, again and again. Detect it and tell the model.

In [ ]:
from collections import Counter
seen = Counter()
def run_guarded(name, args):
    key = (name, json.dumps(args, sort_keys=True))
    seen[key] += 1
    if seen[key] >= 3:
        return f"You already called {name} with this input {seen[key] - 1} times. Use the earlier result.", True
    return dispatch(name, args, session)

for _ in range(3):  # simulate a stuck agent without calling the API
    print(run_guarded("get_order", {"order_id": "O2001"})[0][:80])

## 6. Compare with the course helper
`course_utils.run_agent` is the same loop, packaged. You'll use it from Week 2 on.

In [ ]:
from course_utils import run_agent
fns = {t["name"]: (lambda n: lambda **a: dispatch(n, a, session)[0])(t["name"]) for t in TOOLS}
res = run_agent("Where is my smart watch?", TOOLS, fns, system=SYSTEM, thinking_level="low")
print(res.stop_reason, res.steps, "steps,", len(res.tool_calls), "tool calls")

## ✏️ Exercises
1. Set `max_steps=2` and give a multi-step task. What does the customer see? How would you make that message better?
2. Implement **plan-and-execute**: ask MODEL for a numbered plan (structured output), then run each step with CHEAP_MODEL. Compare cost with ReAct.
3. Add a 4th budget: max tool calls per tool name.

## 🧾 What did this notebook cost?
Every call above was recorded by `tracker` (estimated at paid prices; the free tier costs $0). Make a habit of checking tokens and cost after every experiment.

In [ ]:
print(tracker.summary())
tracker.to_dataframe().tail(10) if tracker.calls else None